# The Same RAG Pipeline, Rebuilt with LangChain

Notebooks `01` and `02` built a full RAG pipeline by hand: reading the PDF ourselves,
writing our own chunking function, calling OpenAI's embedding API directly, and calling
ChromaDB directly.

**This notebook does the exact same thing**, but using **LangChain** — a library that
provides ready-made building blocks for each of those steps, plus a way to connect the
blocks together called **LCEL** (LangChain Expression Language).

**The one new idea in this notebook: the pipe operator `|`.**
LCEL lets you connect pieces like `prompt | llm | output_parser`. This means: "take the
output of `prompt`, feed it into `llm`, take *that* output, feed it into `output_parser`".
It's the same as calling `output_parser(llm(prompt(input)))`, just written left-to-right.

This notebook saves its data to its **own** folder (`chroma_db_langchain/`), so it never
touches the database built by notebook `01`.

## 1. Setup

Same `.env` file as the other notebooks. LangChain's OpenAI classes read
`OPENAI_API_KEY` from the environment automatically — we don't even need to construct an
OpenAI client ourselves this time.

In [2]:
from dotenv import load_dotenv
load_dotenv(r"E:\Imarticus_ Learning\Jobs - 2026\Models\.env")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## 2. Settings

In [6]:
PDF_PATH = r"E:\Imarticus_ Learning\Jobs - 2026\Models\data\raw\[ML] Introduction to Machine Learning with Python (2017).pdf"

CHROMA_PATH = "../chroma_db_langchain"   # separate folder from notebook 01's chroma_db/
COLLECTION_NAME = "ml_book_langchain"

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-nano"

CHUNK_SIZE = 1000     # characters per chunk
CHUNK_OVERLAP = 200   # characters shared with the previous chunk

## 3. Load the PDF

`PyPDFLoader` replaces our manual `PdfReader` + `extract_text()` loop from notebook 01.
It returns one LangChain `Document` per page — each `Document` bundles the page's text
(`.page_content`) together with its metadata (e.g. `.metadata["page"]`).

In [7]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages.")
pages[0]

Loaded 392 pages.


Document(metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'moddate': '2016-09-22T10:41:19-04:00', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'source': 'E:\\Imarticus_ Learning\\Jobs - 2026\\Models\\data\\raw\\[ML] Introduction to Machine Learning with Python (2017).pdf', 'total_pages': 392, 'page': 0, 'page_label': 'Cover'}, page_content='Andreas C. Müller & Sarah Guido\nIntroduction to \nMachine \nLearning  \nwith P y t h o n   \nA GUIDE FOR DATA SCIENTISTS')

## 4. Split into chunks

`RecursiveCharacterTextSplitter` replaces our hand-written `split_into_chunks()` function.
It splits each page's text into overlapping pieces, and automatically carries each page's
metadata over to its chunks.

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(pages)

print(f"Created {len(chunks)} chunks from {len(pages)} pages.")
print(f"\nExample chunk (from page {chunks[10].metadata['page']}):\n")
print(chunks[10].page_content[:300])

Created 973 chunks from 392 pages.

Example chunk (from page 5):

Linear Models                                                                                                               45
Naive Bayes Classifiers                                                                                                 68
Decision Trees                                    


## 5. Embeddings + vector store, in one step

`OpenAIEmbeddings` replaces our own `get_embeddings()` function — it knows how to call the
OpenAI embeddings API for us.

`Chroma.from_documents(...)` replaces *all* of: creating a `PersistentClient`, creating a
collection, embedding every chunk, and calling `collection.add(...)`. LangChain does all
four steps for us in a single call.

In [9]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH,
)

print(f"Saved {vectorstore._collection.count()} chunks to '{COLLECTION_NAME}'.")

Saved 973 chunks to 'ml_book_langchain'.


## 6. The retriever — LangChain's version of `collection.query()`

A **retriever** is just an object you can hand a question to, and it gives back the most
relevant chunks. It wraps the "embed the question, then search Chroma" logic we wrote by
hand in notebook 02 into a single reusable object — and importantly, a retriever is itself
a chain piece we can pipe into other things (next section).

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

retriever.invoke("What is supervised learning?")

[Document(id='cf9c0a90-0810-4967-8299-57cc7a67e535', metadata={'page': 15, 'moddate': '2016-09-22T10:41:19-04:00', 'total_pages': 392, 'author': 'Andreas C. Müller and Sarah Guido', 'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'page_label': '2', 'title': 'Introduction to Machine Learning with Python', 'creationdate': '2016-09-21T13:04:39+00:00', 'source': 'E:\\Imarticus_ Learning\\Jobs - 2026\\Models\\data\\raw\\[ML] Introduction to Machine Learning with Python (2017).pdf', 'trapped': '/False'}, page_content='learning algorithms because a “teacher” provides supervision to the algorithms in the\nform of the desired outputs for each example that they learn from. While creating a\ndataset of inputs and outputs is often a laborious manual process, supervised learning\nalgorithms are well understood and their performance is easy to measure. If your\napplication can be formulated as 

## 7. The prompt template — LangChain's version of `build_prompt()`

`{context}` and `{question}` are placeholders. When the chain runs, LangChain fills them
in with real values.

In [11]:
prompt = ChatPromptTemplate.from_template("""Answer the question using ONLY the context below.
If the context doesn't contain the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:""")

## 8. The chat model — LangChain's version of `client.chat.completions.create()`

In [12]:
llm = ChatOpenAI(model=CHAT_MODEL)

## 9. A small helper: turn retrieved chunks into one block of text

The retriever gives us a *list* of `Document` objects, but our prompt's `{context}`
placeholder needs a single string. This function joins them together.

In [13]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## 10. Build the chain: `retriever | prompt | llm | output_parser`

This is the LCEL pipe syntax in action. Reading the chain below top to bottom:

1. `{"context": retriever | format_docs, "question": RunnablePassthrough()}` — takes the
   incoming question, sends a **copy** of it through `retriever | format_docs` to build the
   `context` string, and passes another copy straight through unchanged as `question`
   (that's what `RunnablePassthrough` means: "don't change this, just pass it along").
2. `| prompt` — plugs `context` and `question` into the prompt template from Step 7.
3. `| llm` — sends the filled-in prompt to `gpt-4.1-nano` and gets a response back.
4. `| StrOutputParser()` — the model's response is a rich `AIMessage` object; this pulls
   out just the plain text string.

In [14]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 11. Try it!

`chain.invoke(question)` runs the question through every piece of the pipe, in order, and
returns the final answer string.

In [15]:
answer = chain.invoke("What is supervised learning?")
print(answer)

Supervised learning is a type of machine learning where a "teacher" provides supervision to the algorithms in the form of desired outputs for each example they learn from. The goal is to build a model that can predict outcomes for new data based on input/output pairs in the training set.


In [16]:
# Try your own question here
answer = chain.invoke("What is the difference between classification and regression?")
print(answer)

Classification involves predicting a class label from a predefined list of possibilities, such as distinguishing between spam and not spam emails or identifying iris species. Regression involves predicting a continuous value, such as income or RAM prices, where the output can be any number within a range. The main difference is that classification predicts categories, while regression predicts continuous outcomes.


## Recap — same steps, different tools

| Step | Notebooks 01 / 02 (by hand) | This notebook (LangChain) |
|---|---|---|
| Read the PDF | `PdfReader` + `.extract_text()` loop | `PyPDFLoader` |
| Split into chunks | our own `split_into_chunks()` | `RecursiveCharacterTextSplitter` |
| Embed text | our own `get_embeddings()` | `OpenAIEmbeddings` |
| Store + save to disk | `PersistentClient` + `collection.add()` | `Chroma.from_documents()` |
| Search for relevant chunks | `collection.query()` | `retriever` (`vectorstore.as_retriever()`) |
| Build the prompt | an f-string in `build_prompt()` | `ChatPromptTemplate` |
| Call the model | `client.chat.completions.create()` | `ChatOpenAI` |
| Extract the answer text | `response.choices[0].message.content` | `StrOutputParser()` |
| Run all the steps in order | calling each function one by one | `chain = {...} \| prompt \| llm \| StrOutputParser()` |

Same pipeline, same result — LangChain just gives standard names and a `\|` syntax to the
steps we already understand from building them by hand.